# Data Science In Action



# Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportions_ztest

## Data Load

In [ ]:
cahoot_2015 = pd.read_csv("EPD_data/EugeneCAD2015noloc.csv", low_memory=False)
cahoot_2016 = pd.read_csv("EPD_data/EugeneCAD2016noloc.csv", low_memory=False)
cahoot_2017 = pd.read_csv("EPD_data/EugeneCAD2017noloc.csv", low_memory=False)
cahoot_2018 = pd.read_csv("EPD_data/EugeneCAD2018noloc.csv", low_memory=False)
cahoot_2019 = pd.read_csv("EPD_data/EugeneCAD2019noloc.csv", low_memory=False)
cahoot_2020 = pd.read_csv("EPD_data/EugeneCAD2020noloc.csv", low_memory=False)
cahoot_2021 = pd.read_csv("EPD_data/EugeneCAD2021noloc.csv", low_memory=False)
cahoot_2022 = pd.read_csv("EPD_data/EugeneCAD2022noloc.csv", low_memory=False)
cahoot_2023 = pd.read_csv("EPD_data/EugeneCAD2023noloc.csv", low_memory=False)
cahoot_2024 = pd.read_csv("EPD_data/EugeneCAD2024noloc.csv", low_memory=False)
cahoot_2025 = pd.read_csv("EPD_data/EugeneCAD2025noloc.csv", low_memory=False)

mcls = pd.read_excel("MCSLC.xlsx", sheet_name="Abbrev list")

print("CAHOOTS row counts by year (raw):")
for y, d in [(2015,cahoot_2015),(2016,cahoot_2016),(2017,cahoot_2017),(2018,cahoot_2018),
             (2019,cahoot_2019),(2020,cahoot_2020),(2021,cahoot_2021),(2022,cahoot_2022),
             (2023,cahoot_2023),(2024,cahoot_2024),(2025,cahoot_2025)]:
    print(f"  {y}: {len(d):>7,}  agencies: {sorted(d['agency'].astype(str).str.strip().unique())}")
print(f"\nMCLS rows: {len(mcls):,}")

## Cahoot Clean

In [ ]:
def clean_cahoots(df):
    df = df.copy()
    for col in ["agency", "nature", "closecode", "closed_as", "prime_unit"]:
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip()
    df = df[df["agency"] == "CAHE"].reset_index(drop=True)
    return df

cahoot_2015 = clean_cahoots(cahoot_2015)
cahoot_2016 = clean_cahoots(cahoot_2016)
cahoot_2017 = clean_cahoots(cahoot_2017)
cahoot_2018 = clean_cahoots(cahoot_2018)
cahoot_2019 = clean_cahoots(cahoot_2019)
cahoot_2020 = clean_cahoots(cahoot_2020)
cahoot_2021 = clean_cahoots(cahoot_2021)
cahoot_2022 = clean_cahoots(cahoot_2022)
cahoot_2023 = clean_cahoots(cahoot_2023)
cahoot_2024 = clean_cahoots(cahoot_2024)
cahoot_2025 = clean_cahoots(cahoot_2025)

print("CAHOOTS row counts by year with filter:")
for y, d in [(2015, cahoot_2015),(2016,cahoot_2016),(2017,cahoot_2017),(2018,cahoot_2018),
             (2019,cahoot_2019),(2020,cahoot_2020),(2021,cahoot_2021),(2022,cahoot_2022),
             (2023,cahoot_2023),(2024,cahoot_2024),(2025,cahoot_2025)]:
    print(f"  cahoot_{y}: {len(d):>6,}")

## MCLS Clean

In [ ]:
mcls = mcls[["ID", "End Point of Dispatch", "Reason for Dispatch #1", "Disposition"]].copy()

mcls["End Point of Dispatch"]  = mcls["End Point of Dispatch"].astype("string").str.strip()
mcls["Reason for Dispatch #1"] = mcls["Reason for Dispatch #1"].astype("string").str.strip()
mcls["Disposition"] = mcls["Disposition"].astype("string").str.strip()


print(f"MCLS rows: {len(mcls):,}")
print("Columns:", list(mcls.columns))
print("\nDisposition value counts:")
print(mcls["Disposition"].value_counts(dropna=False))

## Build EPD_handoff - Cahoots

In [ ]:
handoff_values = {
    "REFERRED TO OTHER AGENCY",
    "RELAYED TO LANE COUNTY SHERIFFS OFFICE",
    "RELAYED TO OREGON STATE POLICE",
    "UNIFORM TRAFFIC CITATION ISSUED",
}

def add_cahoots_handoff(df):
    df = df.copy()
    df["epd_handoff"] = df["closed_as"].isin(handoff_values).astype(int)
    return df

df_2022 = add_cahoots_handoff(cahoot_2022)
df_2023 = add_cahoots_handoff(cahoot_2023)
df_2024 = add_cahoots_handoff(cahoot_2024)
df_2025 = add_cahoots_handoff(cahoot_2025)

audit = pd.concat([df_2022, df_2023, df_2024, df_2025], ignore_index=True)
print("CAHOOTS closed_as × epd_handoff (2022–2025 combined):")
print(pd.crosstab(audit["closed_as"], audit["epd_handoff"], margins=True))

## Build EPD_handoff - MCLS

In [ ]:
mcls["epd_handoff"] = (mcls["Disposition"] == "Arrest").astype(int)

print("MCLS Disposition × epd_handoff:")
print(pd.crosstab(mcls["Disposition"], mcls["epd_handoff"], margins=True))
print(f"\nTotal handoffs: {mcls['epd_handoff'].sum():,} of {len(mcls):,} "
      f"({mcls['epd_handoff'].mean():.2%})")

# RQ1 - Handoff Rate Summary

In [ ]:
cahoots_all = pd.concat([df_2022, df_2023, df_2024, df_2025], ignore_index=True)

# Calculate CAHOOTS handoff rate
n_calls_cahoots = len(cahoots_all)
n_handoffs_cahoots = cahoots_all["epd_handoff"].sum()
rate_cahoots = n_handoffs_cahoots / n_calls_cahoots

# Calculate MCLS handoff rate
n_calls_mcls = len(mcls)
n_handoffs_mcls = mcls["epd_handoff"].sum()
rate_mcls = n_handoffs_mcls / n_calls_mcls

summary = pd.DataFrame({
    "prohram" : ["CAHOOTS (2022–2025)", "MCLS"],
    "n_calls": [n_calls_cahoots, n_calls_mcls],
    "n_handoffs": [n_handoffs_cahoots, n_handoffs_mcls],
    "handoff_rate": [rate_cahoots, rate_mcls],
})

summary["handoff_pct"] = (summary["handoff_rate"] * 100).round(2).astype(str) + "%"

print("Handoff rate summary:")
print(summary.to_string(index=False))

In [ ]:
print(mcls.columns.tolist())
print(mcls["epd_handoff"].value_counts())